# RealThor v0.2 — Structured Output with LangChain

In v0.1, RealThor returned free-form text. That worked for reading, but made evaluation brittle — we had to parse strings and match phrases that the model might phrase differently every run.

v0.2 upgrades to **structured output**: instead of a paragraph, RealThor returns a validated Pydantic object with typed fields. Evaluation becomes exact field comparison — no string parsing.

In [ ]:
import os
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
load_dotenv()

In [ ]:
with open("prompts/system_prompt_v2.md") as f:
    SYSTEM_PROMPT = f.read()

In [ ]:
demand = pd.read_csv("data/demand_signals.csv")

In [ ]:
supply = pd.read_csv("data/supply_snapshot.csv")

## Define the Output Schema

This Pydantic model mirrors the output fields in `system_prompt_v2.md`. The model is forced to return values that match these types — no free-form text.

In [ ]:
class ListingRecommendation(BaseModel):
    """Structured recommendation from RealThor."""
    priority: Literal["high", "medium", "low"]
    demand_reasoning: str
    supply_reasoning: str
    recommended_action: Literal["prioritize_new_listings", "monitor_opportunity", "deprioritize"]
    confidence: Literal["high", "medium", "low"]

## Create the Structured LLM

`with_structured_output()` wraps the model so every response is parsed and validated against `ListingRecommendation`. If the model produces an invalid value, LangChain retries automatically.

In [ ]:
provider = os.getenv("LLM_PROVIDER", "openai")
model_kwargs = {"model": "gpt-4o-mini", "temperature": 0}
if provider == "vocareum":
    model_kwargs |= {"base_url": "https://openai.vocareum.com/v1", "api_key": os.getenv("VOCAREUM_API_KEY")}
llm = ChatOpenAI(**model_kwargs)

In [ ]:
structured_llm = llm.with_structured_output(ListingRecommendation)

## Helper Functions

In [ ]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]
    if d.empty or s.empty:
        return "No data available for the requested segment."
    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [ ]:
def ask_realthor_v2(region: str, property_type: str, bedrooms: int) -> ListingRecommendation:
    context = build_context(region, property_type, bedrooms)
    return structured_llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=context),
    ])

## Run One Example

In [ ]:
result = ask_realthor_v2("west", "apartment", 2)

In [ ]:
print(f"Priority:           {result.priority}")
print(f"Recommended action: {result.recommended_action}")
print(f"Confidence:         {result.confidence}")
print(f"\nDemand reasoning:   {result.demand_reasoning}")
print(f"Supply reasoning:   {result.supply_reasoning}")

## What Changed

| | v0.1 | v0.2 |
|---|---|---|
| Output | Free-form string | Pydantic object |
| Priority check | `"high" in output.lower()` | `result.priority == "high"` |
| Action check | String search | `result.recommended_action == "prioritize_new_listings"` |
| Invalid values | Model can say anything | LangChain/Pydantic validates and retries |
| Evaluation | Brittle | More deterministic |

From this point forward, all evaluation uses `result.priority` and `result.recommended_action` — exact field comparisons against the golden dataset.